<a href="https://colab.research.google.com/github/AllyApitchaya/msc-adr-prediction/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/MSc_ADR_Project'
print(f"Project path: {PROJECT_PATH}")

Mounted at /content/drive
Project path: /content/drive/MyDrive/MSc_ADR_Project


In [2]:
import pandas as pd
import os

# Load the combined drug-ADR dataset produced by notebook 01
data_path = os.path.join(PROJECT_PATH, 'data', 'processed',
                         'drug_adr_with_structures.csv')
df = pd.read_csv(data_path)

print(f"Loaded {len(df):,} drug-ADR pairs")
print(f"Columns: {list(df.columns)}\n")
print(f"Unique drugs: {df['drug'].nunique()}")
print(f"Unique ADRs:  {df['adr'].nunique():,}")
print(f"Unique cases: {df['caseid'].nunique():,}")
print(f"Quarters:     {sorted(df['quarter'].unique())}")

Loaded 80,958 drug-ADR pairs
Columns: ['drug', 'pubchem_cid', 'smiles', 'adr', 'caseid', 'quarter']

Unique drugs: 15
Unique ADRs:  3,586
Unique cases: 23,218
Quarters:     ['2020q1', '2020q2', '2020q3', '2020q4', '2021q1', '2021q2', '2021q3', '2021q4', '2022q1', '2022q2', '2022q3', '2022q4']


In [3]:
# Before removing anything, examine the ADR terms.
# FAERS 'pt' values include genuine adverse reactions but also
# medication errors, the underlying disease, and report-quality
# terms that are not true ADRs.

adr_counts = df['adr'].value_counts()

print(f"Total unique ADR terms: {len(adr_counts):,}\n")
print("Top 40 most frequent terms (inspect for non-ADR noise):")
print(adr_counts.head(40).to_string())

Total unique ADR terms: 3,586

Top 40 most frequent terms (inspect for non-ADR noise):
adr
Lactic acidosis                         4312
Acute kidney injury                     2655
Metabolic acidosis                      1703
Diarrhoea                               1646
Toxicity to various agents              1493
Hypoglycaemia                           1208
Drug ineffective                        1010
Nausea                                  1006
Vomiting                                 983
Blood glucose increased                  679
Fatigue                                  663
Hypotension                              660
Diabetic ketoacidosis                    655
Overdose                                 640
Euglycaemic diabetic ketoacidosis        582
Dyspnoea                                 581
Hyperkalaemia                            570
Weight decreased                         545
Off label use                            538
Renal failure                            532
Malaise  

In [4]:
# Non-ADR terms to remove. FAERS 'pt' values include medication
# errors, drug-efficacy complaints, the underlying disease, broad
# non-specific terms, and outcomes. These are not adverse drug
# reactions and are excluded before modelling.
NON_ADR_TERMS = {
    # Medication errors / misuse
    'Off label use',
    'Overdose',
    'Intentional overdose',
    'Drug interaction',
    # Drug-efficacy complaints (not reactions)
    'Drug ineffective',
    # Underlying disease being treated
    'Diabetes mellitus inadequate control',
    # Non-specific / uninformative terms
    'Toxicity to various agents',
    'Condition aggravated',
    # Outcomes rather than specific reactions
    'Death',
    'Completed suicide',
}

before = len(df)
df_clean = df[~df['adr'].isin(NON_ADR_TERMS)].copy()
removed = before - len(df_clean)

print(f"Before cleaning: {before:,} drug-ADR pairs")
print(f"Removed:         {removed:,} non-ADR pairs "
      f"({removed / before * 100:.1f}%)")
print(f"After cleaning:  {len(df_clean):,} drug-ADR pairs\n")

print(f"Unique ADR terms: {df_clean['adr'].nunique():,} "
      f"(was {df['adr'].nunique():,})")
print(f"Unique cases:     {df_clean['caseid'].nunique():,}")
print("\nTop 15 ADRs after cleaning:")
print(df_clean['adr'].value_counts().head(15).to_string())

Before cleaning: 80,958 drug-ADR pairs
Removed:         6,130 non-ADR pairs (7.6%)
After cleaning:  74,828 drug-ADR pairs

Unique ADR terms: 3,576 (was 3,586)
Unique cases:     22,144

Top 15 ADRs after cleaning:
adr
Lactic acidosis                      4312
Acute kidney injury                  2655
Metabolic acidosis                   1703
Diarrhoea                            1646
Hypoglycaemia                        1208
Nausea                               1006
Vomiting                              983
Blood glucose increased               679
Fatigue                               663
Hypotension                           660
Diabetic ketoacidosis                 655
Euglycaemic diabetic ketoacidosis     582
Dyspnoea                              581
Hyperkalaemia                         570
Weight decreased                      545


In [5]:
# Temporal split following the project proposal's intent to
# simulate real-world deployment: the model is trained on past
# reports and evaluated on future ones. A random split would
# leak future information into training.
#
# Splitting by quarter also prevents case-level leakage: each
# FAERS case belongs to a single quarter, so all drug-ADR pairs
# from one case stay within the same split.

TRAIN_QUARTERS = ['2020q1', '2020q2', '2020q3',
                  '2020q4', '2021q1', '2021q2']
VAL_QUARTERS   = ['2021q3', '2021q4']
TEST_QUARTERS  = ['2022q1', '2022q2', '2022q3', '2022q4']

train_df = df_clean[df_clean['quarter'].isin(TRAIN_QUARTERS)].copy()
val_df   = df_clean[df_clean['quarter'].isin(VAL_QUARTERS)].copy()
test_df  = df_clean[df_clean['quarter'].isin(TEST_QUARTERS)].copy()

total = len(df_clean)
print("Temporal split (by quarter):\n")
for name, subset, quarters in [
    ('Train', train_df, TRAIN_QUARTERS),
    ('Val',   val_df,   VAL_QUARTERS),
    ('Test',  test_df,  TEST_QUARTERS),
]:
    pct = len(subset) / total * 100
    print(f"  {name:6s} {quarters[0]}-{quarters[-1]}: "
          f"{len(subset):>7,} pairs ({pct:4.1f}%)  "
          f"{subset['caseid'].nunique():>6,} cases")

# Verify no case appears in more than one split
train_cases = set(train_df['caseid'])
val_cases   = set(val_df['caseid'])
test_cases  = set(test_df['caseid'])

overlap = ((train_cases & val_cases) |
           (train_cases & test_cases) |
           (val_cases & test_cases))
print(f"\nCase overlap between splits: {len(overlap)} "
      f"({'OK - no leakage' if not overlap else 'WARNING'})")

Temporal split (by quarter):

  Train  2020q1-2021q2:  36,766 pairs (49.1%)  10,783 cases
  Val    2021q3-2021q4:  11,790 pairs (15.8%)   3,331 cases
  Test   2022q1-2022q4:  26,272 pairs (35.1%)   8,030 cases

Case overlap between splits: 0 (OK - no leakage)


In [6]:
import numpy as np

# Negative sampling at a 1:1 ratio, following Wan et al. (DeepADR).
#
# Positive pairs are drug-ADR combinations observed in FAERS
# (label 1). Negative pairs are drug-ADR combinations NOT observed
# (label 0), sampled randomly in equal number.
#
# Sampling is done independently per split so that knowledge of
# which pairs are negative does not leak across train/val/test.

RANDOM_SEED = 42  # fixed seed for reproducibility


def add_negative_samples(split_df, all_adrs, seed):
    """Return a balanced DataFrame of positive and negative
    drug-ADR pairs for one split.
    """
    rng = np.random.default_rng(seed)

    # Positive pairs: unique (drug, adr) combinations in this split
    pos = (split_df[['drug', 'adr']]
           .drop_duplicates()
           .reset_index(drop=True))
    pos_set = set(zip(pos['drug'], pos['adr']))

    # SMILES lookup so negatives carry the same drug features
    smiles_map = (split_df.drop_duplicates('drug')
                  .set_index('drug')['smiles'].to_dict())
    cid_map = (split_df.drop_duplicates('drug')
               .set_index('drug')['pubchem_cid'].to_dict())

    drugs = list(smiles_map.keys())

    # Sample negative pairs: random (drug, adr) not seen as positive
    negatives = []
    n_needed = len(pos)
    attempts = 0
    max_attempts = n_needed * 50  # safety limit

    while len(negatives) < n_needed and attempts < max_attempts:
        drug = rng.choice(drugs)
        adr = rng.choice(all_adrs)
        attempts += 1
        if (drug, adr) not in pos_set:
            negatives.append((drug, adr))
            pos_set.add((drug, adr))  # avoid duplicate negatives

    neg = pd.DataFrame(negatives, columns=['drug', 'adr'])

    # Attach labels and drug features
    pos = pos.assign(label=1)
    neg = neg.assign(label=0)
    combined = pd.concat([pos, neg], ignore_index=True)
    combined['smiles'] = combined['drug'].map(smiles_map)
    combined['pubchem_cid'] = combined['drug'].map(cid_map)

    # Shuffle so positives and negatives are interleaved
    combined = combined.sample(frac=1, random_state=seed)
    return combined.reset_index(drop=True)


# The ADR vocabulary is shared (all ADRs seen anywhere in the data)
all_adrs = df_clean['adr'].unique()

train_final = add_negative_samples(train_df, all_adrs, RANDOM_SEED)
val_final   = add_negative_samples(val_df,   all_adrs, RANDOM_SEED + 1)
test_final  = add_negative_samples(test_df,  all_adrs, RANDOM_SEED + 2)

print("Balanced datasets (positive + negative):\n")
for name, subset in [('Train', train_final),
                     ('Val',   val_final),
                     ('Test',  test_final)]:
    n_pos = (subset['label'] == 1).sum()
    n_neg = (subset['label'] == 0).sum()
    print(f"  {name:6s} {len(subset):>7,} pairs  "
          f"(positive: {n_pos:>6,}  negative: {n_neg:>6,})")

Balanced datasets (positive + negative):

  Train    8,240 pairs  (positive:  4,120  negative:  4,120)
  Val      4,912 pairs  (positive:  2,456  negative:  2,456)
  Test     9,016 pairs  (positive:  4,508  negative:  4,508)


In [7]:
import os

# Save the preprocessed datasets. These are model-ready:
# balanced, labelled, with SMILES attached, and split temporally.
processed_dir = os.path.join(PROJECT_PATH, 'data', 'processed')

splits = {
    'train': train_final,
    'val':   val_final,
    'test':  test_final,
}

print("Saving preprocessed datasets:\n")
for name, subset in splits.items():
    # Order columns consistently
    cols = ['drug', 'pubchem_cid', 'smiles', 'adr', 'label']
    subset = subset[cols]
    out_path = os.path.join(processed_dir, f'{name}.csv')
    subset.to_csv(out_path, index=False)
    print(f"  {name:6s} -> {out_path}")
    print(f"         {len(subset):,} pairs, "
          f"{subset['label'].mean() * 100:.0f}% positive")

print("\nSample of train.csv:")
print(train_final[['drug', 'adr', 'label']].head(10).to_string(index=False))

Saving preprocessed datasets:

  train  -> /content/drive/MyDrive/MSc_ADR_Project/data/processed/train.csv
         8,240 pairs, 50% positive
  val    -> /content/drive/MyDrive/MSc_ADR_Project/data/processed/val.csv
         4,912 pairs, 50% positive
  test   -> /content/drive/MyDrive/MSc_ADR_Project/data/processed/test.csv
         9,016 pairs, 50% positive

Sample of train.csv:
         drug                          adr  label
    glipizide                 Constipation      1
  nateglinide Hyperplastic cholecystopathy      0
    metformin        Hepatitis cholestatic      1
    glyburide          Portal hypertension      0
    glyburide                  Hypotension      1
dapagliflozin                       Chorea      0
  saxagliptin                     Vulvitis      0
     acarbose               Penis disorder      0
    metformin     Oral mucosal exfoliation      1
  repaglinide         Fat tissue increased      1
